# Stage 0 - stage the enrichment-candidate datasets

Phase 5.54 Stage 0: pull every candidate dataset into a read-only, tidy cache
under `data/_raw/lexicon/staging/` so the topic notebooks read clean inputs.

This notebook is **automatic**: it downloads and stages everything (Tatoeba,
frequency, the Wikidata CC0 lexeme dump, the Kelly CEFR lists) and writes one
`_staging.json` manifest. All logic lives in
`lang_tools.lexicon.ingestion.staging`; the cells only wire the calls.

Prerequisites: `uv sync --extra enrich --extra ingest --extra store` (wordfreq +
xlrd for frequency / CEFR). Note the Wikidata lexeme dump is ~590 MB - the first
run is slow; reruns skip it. Nothing here writes the source-of-truth Parquet.


In [ ]:
from loguru import logger as lg

from lang_tools.lexicon.ingestion import staging
from lang_tools.params.lang_tools_params import get_lang_tools_params

LANGS = ["en", "pt", "es", "fr", "it"]
data_fol = get_lang_tools_params().paths.data_fol
records: list[staging.StagedDataset] = []
lg.info("00_stage: langs={} -> staging under {}", LANGS, staging.staging_dir(data_fol))

## OMW + CILI (already staged)

The backbone and the English-gloss resource are downloaded by
`acquire.download_omw` (see `01_download`). Just record them in the manifest so
the full input set and its licenses live in one place.


In [ ]:
records += staging.omw_cili_staged_records(LANGS)
records[-2:]

## Tatoeba example sentences (network, CC-BY)

Downloads the per-language sentence export and stages `(sentence_id, text)`.
The lemma join is sense-blind, so this is for examples only.


In [ ]:
records += staging.download_tatoeba_sentences(LANGS, data_fol=data_fol)
records[-len(LANGS) :]

## Token frequency (`enrich` extra: `wordfreq`)

Per-language top-N word forms with rank + zipf - the language-level frequency
signal for Topic 3. Needs `uv sync --extra enrich`.


In [ ]:
records += [staging.stage_frequency_list(lang, data_fol=data_fol) for lang in LANGS]
records[-len(LANGS) :]

## Wikidata lexemes (CC0)

The public SPARQL endpoint is throttled (a global count returns HTTP 429, ~1
req/min), so the viable source is the **CC0 lexeme dump** (`latest-lexemes.json.gz`,
~590 MB, no rate limit, exact counts). Download it once into the path below, then
stage per language. A gentle SPARQL sample probe stays available as a fallback.


In [ ]:
# CC0 lexeme dump (the rate-limit-free source). Downloads ~590 MB on first run,
# skips if already present, then partitions per language with exact counts.
dump_path = staging.download_lexeme_dump(data_fol=data_fol)
records += staging.stage_wikidata_lexeme_dump(
    LANGS, data_fol=data_fol, dump_path=dump_path
)
print(records[-len(LANGS) :])

## Graded / CEFR list (validation only)

No clean permissive multilingual list exists, so Kelly (en/it) is the realistic
download - `.xls`, CC-BY-NC-SA, read directly via `xlrd`. Non-commercial /
share-alike is fine: the list is validation-only and never merged. pt/es/fr have
no graded list (estimate in phase 6); Oxford is PDF/guidance-only. The grounded
candidates and how to read each live in `KNOWN_CEFR_SOURCES`.


In [ ]:
# Download + parse the real graded lists. Kelly (en/it) are .xls (CC-BY-NC-SA),
# read via xlrd; validation-only, never shipped. pt/es/fr have no graded list
# (estimate in phase 6); Oxford is PDF/guidance-only (not auto-downloadable).
for src_name in ("kelly-en", "kelly-it"):
    records.append(staging.download_cefr_source(src_name, data_fol=data_fol))
records[-2:]

## Write the staging manifest

Records every staged dataset (source, version, license, row count) for the
phase-10 licensing audit.


In [ ]:
manifest_path = staging.write_staging_manifest(data_fol, records)
lg.success("Wrote staging manifest: {} ({} datasets)", manifest_path, len(records))
staging.read_staging_manifest(data_fol)

## Optional: Tatoeba lemma -> sentence index

Build the sense-blind example index for one language against the corpus lemmas.
Needs a built corpus under `data/lexicon/` (run `02_transform` first).


In [ ]:
import pyarrow.parquet as pq

from lang_tools.lexicon.lemma_store import LexiconStore

LANG = "en"
table = pq.read_table(staging.staging_dir(data_fol) / "tatoeba" / f"{LANG}.parquet")
sentences = list(
    zip(
        table.column("sentence_id").to_pylist(),
        table.column("text").to_pylist(),
        strict=False,
    )
)

store = LexiconStore.from_data_fol(data_fol)
forms = [lem.text for lem in store.get_lemmas_by_language(LANG)]
index = staging.build_lemma_sentence_index(sentences, forms)
lg.info("Indexed {} {} lemma forms to Tatoeba sentences", len(index), LANG)
sorted(index.items(), key=lambda kv: -len(kv[1]))[:10]